## Open notebook in:
| Colab                               Gradient                                                                                                                                         |
|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/Transformers-in-Action/blob/main/CH03/ch_03_code_examples.ipynb)                                              


# About the Notebook

This notebook consolidates the practical code examples from **Chapter 3: Model Families and Architecture Variants**. It serves as a hands-on companion to the text, allowing you to explore and experiment with the core transformer architectures discussed in the chapter.

**Purpose**
It provides runnable, minimal implementations of:

* Encoder-only and decoder-only transformer blocks
* Masked attention and causal masking
* Masked language modeling (MLM) heads
* Sentence embedding generation and similarity search
* Mixture of Experts (MoE) gating mechanisms

**How to Use It**
Each section can be executed independently for exploration. All models are simplified for educational purposes and annotated for clarity rather than optimized for production.




# Imports

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# 1) Minimal BPE tokenizer (Unicode‑safe)

In [ ]:
# Adapted from OpenAI GPT‑2 tokenizer ideas, simplified for clarity.
# This is not a complete drop‑in replacement for production tokenizers.


def bytes_to_unicode() -> Dict[int, str]:
    """Map bytes to a Unicode string so we can represent arbitrary bytes as text tokens.
    Mirrors the approach used in GPT‑2 to keep reversibility.
    """
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))


def get_pairs(word: Tuple[str, ...]) -> set:
    """Return set of symbol pairs in a word.
    word is a tuple of symbols (symbols can be characters or merged strings).
    """
    pairs = set()
    prev_char = word[0]
    for ch in word[1:]:
        pairs.add((prev_char, ch))
        prev_char = ch
    return pairs


class BPETokenizer:
    """Tiny Byte Pair Encoding tokenizer used for the decoder‑only input example.

    Parameters
    ----------
    encoder : Dict[str, int]
        Vocabulary mapping from string token to id.
    bpe_merges : List[Tuple[str, str]]
        Merge rules in priority order.
    errors : str
        Error handling for bytes decode.
    """

    def __init__(self, encoder: Dict[str, int], bpe_merges: List[Tuple[str, str]], errors: str = "replace"):
        self.encoder = encoder
        self.decoder = {v: k for k, v in encoder.items()}
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}
        self.bpe_ranks = {merge: i for i, merge in enumerate(bpe_merges)}
        self.cache: Dict[str, str] = {}
        self.errors = errors
        # Regex to split on common patterns (letters, numbers, punctuation, whitespace runs)
        self.pat = re.compile(
            r"''s|''t|''re|''ve|''m|''ll|''d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+",
            re.UNICODE,
        )

    def bpe(self, token: str) -> str:
        if token in self.cache:
            return self.cache[token]
        word = tuple(token)
        pairs = get_pairs(word)
        if not pairs:
            return token

        while True:
            bigram = min(pairs, key=lambda p: self.bpe_ranks.get(p, 10**10))
            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            new_word: List[str] = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                except ValueError:
                    new_word.extend(word[i:])
                    break
                new_word.extend(word[i:j])
                i = j
                if i < len(word) - 1 and word[i] == first and word[i + 1] == second:
                    new_word.append(first + second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            word = tuple(new_word)
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)
        out = " ".join(word)
        self.cache[token] = out
        return out

    def encode(self, text: str) -> List[int]:
        bpe_tokens: List[int] = []
        # Split text into initial candidates
        for token in re.findall(self.pat, text):
            # Map to byte‑safe Unicode
            byte_seq = "".join(self.byte_encoder[b] for b in token.encode("utf-8"))
            # Apply BPE merges
            for bpe_token in self.bpe(byte_seq).split(" "):
                if bpe_token not in self.encoder:
                    raise KeyError(f"Unknown token piece '{bpe_token}' — vocab incomplete.")
                bpe_tokens.append(self.encoder[bpe_token])
        return bpe_tokens

    def decode(self, tokens: Iterable[int]) -> str:
        text = "".join([self.decoder[t] for t in tokens])
        out = bytearray([self.byte_decoder[c] for c in text]).decode("utf-8", errors=self.errors)
        return out

    @staticmethod
    def from_files(model_dir: str, model_name: str) -> "BPETokenizer":
        with open(os.path.join(model_dir, model_name, "encoder.json"), "r", encoding="utf-8") as f:
            encoder = json.load(f)
        with open(os.path.join(model_dir, model_name, "vocab.bpe"), "r", encoding="utf-8") as f:
            merges_data = f.read().splitlines()
        # Skip header
        merges = [tuple(line.split()) for line in merges_data if line and not line.startswith("#")]
        return BPETokenizer(encoder=encoder, bpe_merges=merges)


# 2) Causal mask and masked attention (PyTorch)

In [ ]:
def causal_mask(sz_q: int, sz_k: int, device: Optional[torch.device] = None, dtype: torch.dtype = torch.bool) -> torch.Tensor:
    """Create an upper‑triangular mask that prevents attending to future positions.
    Returns shape [1, 1, sz_q, sz_k] for easy broadcasting over [B, H, Q, K].
    """
    i = torch.arange(sz_q, device=device).unsqueeze(1)
    j = torch.arange(sz_k, device=device).unsqueeze(0)
    m = (i >= j - (sz_k - sz_q)).to(dtype)
    return m.view(1, 1, sz_q, sz_k)


def apply_causal_mask(attn_weights: torch.Tensor) -> torch.Tensor:
    """Apply a causal mask to attention weights.
    attn_weights: [B, H, Q, K]
    """
    B, H, Q, K = attn_weights.shape
    mask = causal_mask(Q, K, device=attn_weights.device, dtype=torch.bool)
    attn_weights = attn_weights.masked_fill(~mask, float("-inf"))
    return attn_weights


# 3) Decoder‑style Transformer block with optional KV cache

In [ ]:
@dataclass
class KVCache:
    keys: torch.Tensor  # [B, H, K, D]
    vals: torch.Tensor  # [B, H, K, D]


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_head: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_head == 0
        self.d_model = d_model
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,  # [B, T, C]
        kv_cache: Optional[KVCache] = None,
    ) -> Tuple[torch.Tensor, KVCache]:
        B, T, C = x.shape
        qkv = self.qkv(x)  # [B, T, 3C]
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)  # [B, H, T, Dh]
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)

        if kv_cache is not None:
            k = torch.cat([kv_cache.keys, k], dim=2)
            v = torch.cat([kv_cache.vals, v], dim=2)
        new_cache = KVCache(keys=k, vals=v)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)  # [B, H, T, K]
        attn_scores = apply_causal_mask(attn_scores)
        attn = F.softmax(attn_scores, dim=-1)
        attn = self.dropout(attn)
        y = torch.matmul(attn, v)  # [B, H, T, Dh]
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.o(y)
        return y, new_cache


class FeedForward(nn.Module):
    def __init__(self, d_model: int, mult: int = 4, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, mult * d_model),
            nn.GELU(),
            nn.Linear(mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_head, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, mult=4, dropout=dropout)

    def forward(self, x: torch.Tensor, kv_cache: Optional[KVCache] = None) -> Tuple[torch.Tensor, KVCache]:
        a, kv_cache = self.attn(self.ln1(x), kv_cache)
        x = x + a
        m = self.ff(self.ln2(x))
        x = x + m
        return x, kv_cache


# 4) Minimal encoder‑only transformer (no causal mask, bidirectional attention)


In [ ]:

class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_head, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, mult=4, dropout=dropout)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        y, _ = self.self_attn(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.ln1(x + y)
        y = self.ff(x)
        x = self.ln2(x + y)
        return x


class EncoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 512, n_head: int = 8, n_layer: int = 6, max_len: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_head, dropout) for _ in range(n_layer)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, src_tokens: torch.Tensor, src_pad_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # src_tokens: [B, T]
        B, T = src_tokens.shape
        pos = torch.arange(T, device=src_tokens.device).unsqueeze(0).expand(B, T)
        x = self.tok_embed(src_tokens) + self.pos_embed(pos)
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, key_padding_mask=src_pad_mask)
        return x  # [B, T, C]



# 5) Masked Language Modeling head


In [ ]:
class MaskedLMHead(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.dense = nn.Linear(d_model, d_model)
        self.activation = nn.GELU()
        self.ln = nn.LayerNorm(d_model)
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, hidden_states: torch.Tensor, masked_positions: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Project hidden states to vocab logits.
        If masked_positions is provided (bool mask or index tensor), select only those.
        """
        if masked_positions is not None:
            hidden_states = hidden_states[masked_positions]
        x = self.dense(hidden_states)
        x = self.activation(x)
        x = self.ln(x)
        logits = self.proj(x)
        return logits


# 6) Embedding usage and similarity search demo (SentenceTransformers)


In [ ]:
def demo_embeddings() -> None:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
    sentences = [
        "Transformers capture context effectively",
        "Embeddings help understand context",
    ]
    embs = model.encode(sentences)
    print("Embeddings shape:", embs.shape)


def demo_similarity() -> None:
    from sentence_transformers import SentenceTransformer, util

    model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
    queries = [
        "What is an embedding in machine learning?",
        "How are embeddings generated in transformer models?",
    ]
    documents = [
        """Embeddings are vector representations of data, such as words or sentences,
        used to capture semantic relationships in machine learning models.""",
        """Transformer models generate embeddings by passing input tokens through
        multiple layers of attention and feedforward networks, producing contextualized
        vector outputs.""",
    ]
    q = model.encode(queries, prompt_name="query") if hasattr(model, "encode") else model.encode(queries)
    d = model.encode(documents)
    scores = util.cos_sim(torch.tensor(q), torch.tensor(d))
    print("Cosine similarity matrix:\n", scores)


In [ ]:

# If run as a script, execute small demos for embeddings and similarity.
if __name__ == "__main__":
    try:
        demo_embeddings()
        demo_similarity()
    except Exception as e:
        print("Embedding demos skipped:", e)
